# Admission Pattern Analysis

## Project Overview
Analyze applicant and admissions data to explore acceptance patterns by academic profile, region, and intake period.

## Learning Objectives
- Explore admission rates by academic profile and demographics
- Identify seasonal and intake-period trends
- Analyze regional variation in applicant quality and acceptance
- Detect potential biases or imbalances in admission outcomes

## Problem Statement
University enrollment management needs to understand what drives admission decisions, how acceptance rates vary across applicant profiles, and whether intake patterns are shifting over time.

## Why This Project Matters
Admission analytics supports equitable access, enrollment forecasting, and strategic recruitment. Understanding patterns helps institutions align capacity with demand and improve diversity outcomes.

## Dataset Overview
Synthetic admissions dataset: ~5,000 applicants over 4 intake cycles with GPA, test scores, extracurriculars, region, and admission decision.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 5000
gpa = np.clip(np.random.normal(3.2, 0.5, n), 1.5, 4.0).round(2)
test_score = np.clip(np.random.normal(1100, 180, n), 400, 1600).astype(int)
extracurriculars = np.random.choice(['None', 'Low', 'Medium', 'High'], n, p=[0.10, 0.25, 0.40, 0.25])
region = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West', 'International'], n, p=[0.25, 0.20, 0.18, 0.22, 0.15])
intake = np.random.choice(['Fall 2022', 'Spring 2023', 'Fall 2023', 'Spring 2024'], n, p=[0.28, 0.22, 0.28, 0.22])
program = np.random.choice(['Engineering', 'Business', 'Arts & Sciences', 'Health Sciences', 'Education'], n, p=[0.22, 0.25, 0.23, 0.18, 0.12])
gender = np.random.choice(['Male', 'Female', 'Non-Binary'], n, p=[0.48, 0.49, 0.03])
first_gen = np.random.choice(['Yes', 'No'], n, p=[0.30, 0.70])

# Admission probability driven by GPA + test + extracurriculars
extra_score = {'None': 0, 'Low': 0.05, 'Medium': 0.12, 'High': 0.20}
admit_prob = (gpa - 1.5) / 2.5 * 0.5 + (test_score - 400) / 1200 * 0.35 + np.array([extra_score[e] for e in extracurriculars])
admit_prob = np.clip(admit_prob + np.random.normal(0, 0.1, n), 0, 1)
decision = np.where(admit_prob > 0.55, 'Admitted', np.where(admit_prob > 0.40, 'Waitlisted', 'Rejected'))

df = pd.DataFrame({
    'ApplicantID': [f'A{i:05d}' for i in range(n)],
    'GPA': gpa, 'TestScore': test_score, 'Extracurriculars': extracurriculars,
    'Region': region, 'Intake': intake, 'Program': program,
    'Gender': gender, 'FirstGen': first_gen, 'Decision': decision
})
print(f'Dataset: {df.shape}')
print(f'Decision distribution:\n{df["Decision"].value_counts()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nGPA range: {df["GPA"].min()} — {df["GPA"].max()}')
print(f'Test score range: {df["TestScore"].min()} — {df["TestScore"].max()}')

## Overall Admission Rates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['Decision'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Overall Decision Distribution')
axes[0].set_ylabel('')
admit_intake = df.groupby('Intake')['Decision'].value_counts(normalize=True).unstack(fill_value=0)
admit_intake.plot.bar(ax=axes[1], stacked=True, edgecolor='black')
axes[1].set_title('Decision Rates by Intake Period')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'admission_rates.png'), dpi=100, bbox_inches='tight')
plt.show()

## Academic Profile Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Decision', y='GPA', order=['Admitted', 'Waitlisted', 'Rejected'], ax=axes[0])
axes[0].set_title('GPA by Decision')
sns.boxplot(data=df, x='Decision', y='TestScore', order=['Admitted', 'Waitlisted', 'Rejected'], ax=axes[1])
axes[1].set_title('Test Score by Decision')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'academic_profile.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Mean GPA by decision:')
print(df.groupby('Decision')['GPA'].mean().round(2))

## Admission by Program

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
prog_decision = df.groupby(['Program', 'Decision']).size().unstack(fill_value=0)
prog_rate = prog_decision.div(prog_decision.sum(axis=1), axis=0)
prog_rate.plot.bar(ax=ax, stacked=True, edgecolor='black')
ax.set_title('Admission Rate by Program')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'program_admission.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_rate = df.groupby('Region').apply(lambda x: (x['Decision'] == 'Admitted').mean()).sort_values(ascending=False)
region_rate.plot.barh(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Admission Rate by Region')
axes[0].set_xlabel('Admit Rate')
region_gpa = df.groupby('Region')['GPA'].mean().sort_values(ascending=False)
region_gpa.plot.barh(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Mean GPA by Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Extracurricular Impact

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
extra_order = ['None', 'Low', 'Medium', 'High']
extra_rate = df.groupby('Extracurriculars').apply(lambda x: (x['Decision'] == 'Admitted').mean()).reindex(extra_order)
extra_rate.plot.bar(ax=ax, edgecolor='black', color='coral')
ax.set_title('Admission Rate by Extracurricular Level')
ax.set_ylabel('Admit Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'extracurricular_impact.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Admit rate by extracurricular level:')
print(extra_rate.round(3))

## First-Generation and Gender Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fg_rate = df.groupby('FirstGen').apply(lambda x: (x['Decision'] == 'Admitted').mean())
fg_rate.plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Admission Rate: First-Gen vs Non-First-Gen')
axes[0].tick_params(axis='x', rotation=0)
gender_rate = df.groupby('Gender').apply(lambda x: (x['Decision'] == 'Admitted').mean())
gender_rate.plot.bar(ax=axes[1], edgecolor='black', color='teal')
axes[1].set_title('Admission Rate by Gender')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'firstgen_gender.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **GPA and test scores** are the strongest predictors of admission — admitted students have clearly higher academic profiles
- **Extracurricular involvement** has a meaningful positive effect on admission probability
- **Regional variation** exists but is largely explained by differences in applicant academic quality
- **First-generation status** shows slightly lower admission rates — worth monitoring for equity
- **Program-level rates** vary, reflecting different selectivity and applicant pools
- **Intake timing** shows stable admission standards across cycles

## Limitations
- Synthetic data with simplified admission model
- No financial aid, legacy, or athletic recruitment factors
- No essay/interview quality data
- No yield (enrollment after admission) tracking
- No longitudinal student success linkage

## How to Improve This Project
- Add yield analysis (admitted → enrolled conversion)
- Include financial aid and scholarship data
- Build predictive enrollment models
- Add diversity and access index tracking
- Link admission profiles to graduation outcomes

## Production Considerations
- Real-time enrollment dashboards for admissions offices
- Predictive yield models for class size planning
- Equity audits of admission patterns
- Applicant pipeline forecasting by region and program

## Common Mistakes
- Comparing admission rates without controlling for applicant quality
- Ignoring composition effects (e.g., regions with different GPA distributions)
- Treating admission decisions as purely merit-based without context
- Not tracking waitlist conversion rates separately

## Mini Challenge / Exercises
1. What is the GPA threshold that separates most admitted from rejected applicants?
2. Which program has the highest ratio of waitlisted to total applicants?
3. Calculate the admission rate for first-gen international applicants vs non-first-gen domestic.
4. If admission rate needs to increase by 5%, what GPA or test score threshold change would achieve it?

## Final Summary / Key Takeaways
- Admission decisions are primarily driven by GPA and test scores
- Extracurricular participation provides a meaningful boost
- Regional and demographic patterns exist and should be monitored for equity
- Admission analytics enables data-driven enrollment strategy
- Small imbalances in access can compound into significant disparities over multiple intake cycles